# Colab 24f — all-pairs exhaustive evaluation (no server preselection)

**What this is.** 24e selected eval pairs from two supplied server files
(`cath_s20_pairs_sample` + `cath_s20_pairs_high`). Those files are a *sample* (~57k unordered
pairs), and their selection criterion is structural (`src4/src5` = local-alignment scores), which is
**not** our target metric (global normLev). 24f drops the pair files for eval and instead scans the
**entire** filtered pool pairwise with our own normLev — no preselection.

**Model/pools/training are byte-identical to 24e.** Same synthetic AA/SS training, same seeds, same
epochs, same per-feed pools, same encoders. 24f changes *only how the eval set is constructed*. Read
any difference from 24e as the effect of de-biasing the eval, never as the model changing.

**Design decisions (grilled before building):**

1. **AA is a no-op consistency check.** An exhaustive scan already found exactly 5 AA pairs >=0.70 in
   the pool, identical to what the sample carried. So AA here must reproduce 24e's AA numbers — if it
   does not, the new eval code has a bug.
2. **SS / 3Di are the real change.** The sample surfaced 606 / 45 high pairs; exhaustively there are
   **623,077 / 6,009**. The sample also *hub-biased* the SS query set (sampled-query median |T|=434 vs
   representative median |T|=22): sampling pairs and taking endpoints over-represents high-degree
   domains. 24f uses **every** domain that has >=1 true neighbour as a query.
3. **AUROC = exhaustive positives + two negative sets (decision 2c).**
   - *primary* (matches the slide wording "high-similarity vs a random one"): positives = all pairs
     >=0.70, negatives = **all** pairs <0.70 (mostly random -> easy).
   - *stress test*: negatives = mid-band pairs in **[0.30, 0.70)** (hard). Reported alongside so the
     "easy negatives" objection is pre-empted. Computed exhaustively via streaming histograms (no
     subsampling, no 55M-pair memory blow-up).
4. **Retrieval MAP@10 at BOTH bars.** 0.70 (kept as the comparison point, consistent with 24e and the
   presentation) **and** 0.90 (where the SS/3Di task becomes well-posed — the near-tie crowd
   collapses). Same metric, different relevance threshold.
5. **Charts mirror the presentation** (grouped AUROC bars; grouped MAP@10-vs-length bars). The UMAP /
   embedding-space suite is NOT duplicated here — it lives unchanged in 24e.

New files use the `_24f` suffix; 24e and the source files are left untouched
(`memory/feedback_dataset_rollback.md`). **No biological claim is made**; relevance is the exact
global-Levenshtein high-similarity neighbour set.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab24e)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70          # primary high-sim cutoff (positives + retrieval bar A)
BAND_STRICT = 0.90          # well-posed retrieval bar B (SS/3Di)
BAND_MID    = 0.30          # lower edge of the mid-band hard-negative window [0.30, 0.70)
BIN_NAMES = ['far', 'mid', 'high']

SEED_TRAIN_AA = 42
SEED_TRAIN_SS = 43
K_MAP = 10

print(f'high cutoff={BAND_HIGH} | strict bar={BAND_STRICT} | hard-neg band=[{BAND_MID},{BAND_HIGH}) | seed={SEED}')

## 3. Helpers (identical to colab24e)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) > max_len:
        raise ValueError(f'sequence length {len(seq)} exceeds max_len={max_len}')
    idx = [CHAR_TO_IDX[c] for c in seq]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); choices = [c for c in abc if c != s[i]]; s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

## 4. Architecture + training (identical to colab24e)

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K; self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)
    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)

class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)

class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]

def train_encoder(alphabet, band_low, label, train_seed, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed)
    rng = np.random.default_rng(train_seed)
    print(f'\n===== Training {label} encoder (alphabet={alphabet}, band_low={band_low}, seed={train_seed}) =====')
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    labels = np.array([p[2] for p in pairs])
    counts = {b: int(c) for b, c in zip(BIN_NAMES,
              np.bincount([bin_idx_for(l, band_low) for l in labels], minlength=3))}
    print(f'  {len(pairs)} pairs, normLev in [{labels.min():.3f}, {labels.max():.3f}], bins={counts}')
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Per-representation pools (identical to colab24e)

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
raw = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))

RESCUED = {'4z0mC02', '3qkaE02'}

def _valid(seq, is_std, domain):
    return (isinstance(seq, str) and is_std(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or domain in RESCUED))

id_to_aa  = {d: s for d, s in zip(raw['domain_id'],   raw['aa_seq'])            if _valid(s, is_standard_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'],   raw['ss_seq'])            if _valid(s, is_standard_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_standard_aa, d)}

POOL_IDS  = {'AA': list(id_to_aa), 'SS': list(id_to_ss), '3Di': list(id_to_3di)}
LOOKUP    = {'AA': id_to_aa,       'SS': id_to_ss,       '3Di': id_to_3di}
SCORE_COL = {'AA': 'aa_score',     'SS': 'ss_score',     '3Di': '3di_score'}
FEEDS = ['AA', 'SS', '3Di']
for f in FEEDS:
    print(f'  {f:<4} pool = {len(POOL_IDS[f]):>6}')

## 6. Train the two encoders (frozen afterwards)

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)
ENCODERS = [('AA-enc', model_aa), ('SS-enc', model_ss)]

## 7. All-pairs exhaustive oracle

For each representation, compute exact normLev over the **entire** filtered pool (all C(N,2) pairs)
in row-blocks. Produce, per query domain, its true neighbour sets at both bars (>=0.70 and >=0.90).
The query set at a bar = every domain with >=1 true neighbour at that bar (no hub-biased sampling).

In [ ]:
def build_allpairs_oracle(feed, block=1024):
    # Exhaustive: full pool x pool exact Levenshtein, blocked. Returns true-neighbour sets at
    # BAND_HIGH and BAND_STRICT (self excluded), plus degree stats.
    lk = LOOKUP[feed]; ids = POOL_IDS[feed]
    seqs = [lk[d] for d in ids]
    lens = np.array([len(s) for s in seqs]); N = len(ids)
    T_high = {}; T_strict = {}
    deg_high = np.zeros(N, dtype=np.int64); deg_strict = np.zeros(N, dtype=np.int64)
    n_high_ord = 0; n_strict_ord = 0
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLevenshtein.distance, workers=-1).astype(np.float64)
        denom = np.maximum(lens[r0:r1][:, None], lens[None, :]); denom[denom == 0] = 1
        sim = 1.0 - Dm / denom
        rows = np.arange(r1 - r0); sim[rows, np.arange(r0, r1)] = -1.0     # kill diagonal
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a]
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32); deg_high[i] = hi.size
            st = np.where(row >= BAND_STRICT)[0]
            if st.size: T_strict[i] = st.astype(np.int32); deg_strict[i] = st.size
        n_high_ord += int((sim >= BAND_HIGH).sum()); n_strict_ord += int((sim >= BAND_STRICT).sum())
    def _stats(deg):
        nt = deg[deg >= 1]
        return dict(pairs=int(deg.sum()) // 2, queries=int((deg >= 1).sum()),
                    med=int(np.median(nt)) if nt.size else 0,
                    mean=float(nt.mean()) if nt.size else 0.0,
                    mx=int(nt.max()) if nt.size else 0)
    sh, ss = _stats(deg_high), _stats(deg_strict)
    print(f'  [{feed:<3}] N={N:<6} | >=0.70: {sh["pairs"]:>7,} pairs, {sh["queries"]:>6} queries, '
          f'med|T|={sh["med"]:>3} (max {sh["mx"]}) | >=0.90: {ss["pairs"]:>6,} pairs, {ss["queries"]:>5} queries, med|T|={ss["med"]}')
    return dict(ids=ids, N=N, lens=lens, T_high=T_high, T_strict=T_strict,
                stats_high=sh, stats_strict=ss)

print('Building all-pairs exhaustive oracle (this is the ~35s/feed full pool scan)...')
ORACLE = {f: build_allpairs_oracle(f) for f in FEEDS}

## 8. Sample-vs-exhaustive comparison (why 24f differs from 24e)

In [ ]:
# Rebuild what the supplied files surfaced (24e protocol) purely to CONTRAST with exhaustive.
PAIR_COLS = ['domain_a', 'domain_b', 'ss_score', 'aa_score', 'src4', 'src5', 'src_flag']
samp = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'), header=None, names=PAIR_COLS)
high = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'), header=None, names=PAIR_COLS)
src = pd.concat([samp, high], ignore_index=True)
src = src[src['domain_a'] != src['domain_b']]
src['pair_key'] = [frozenset((a, b)) for a, b in zip(src['domain_a'], src['domain_b'])]
src = src.drop_duplicates('pair_key').reset_index(drop=True)

print('=' * 96)
print('SAMPLE (24e supplied files, recomputed normLev)  vs  EXHAUSTIVE (24f all-pairs)  -- high bar 0.70')
print('=' * 96)
print(f'{"feed":<6}{"sample high":>13}{"sample q":>10}{"sample med|T|":>15}'
      f'{"exhaustive high":>18}{"exhaustive q":>14}{"exh med|T|":>12}')
print('-' * 96)
for feed in FEEDS:
    lk = LOOKUP[feed]; poolset = set(POOL_IDS[feed]); idx = {d: i for i, d in enumerate(POOL_IDS[feed])}
    sub = src[src['domain_a'].isin(poolset) & src['domain_b'].isin(poolset)]
    cur = np.array([norm_lev(lk[a], lk[b]) for a, b in zip(sub['domain_a'], sub['domain_b'])])
    hp = sub[cur >= BAND_HIGH]
    sq = sorted(set(hp['domain_a']) | set(hp['domain_b']))
    # sample med|T| uses the SAME full-pool oracle, restricted to sampled queries
    smed = int(np.median([ORACLE[feed]['T_high'][idx[q]].size for q in sq if idx[q] in ORACLE[feed]['T_high']])) if sq else 0
    sh = ORACLE[feed]['stats_high']
    print(f'{feed:<6}{len(hp):>13,}{len(sq):>10,}{smed:>15}{sh["pairs"]:>18,}{sh["queries"]:>14,}{sh["med"]:>12}')
print('-' * 96)
print('Note: SS sampled-query med|T| >> exhaustive med|T| = the hub-bias (sampling pairs over-picks '
      'high-degree domains). AA sample == exhaustive (already complete).')

## 9. Encoder pool embeddings (cache once per encoder x feed)

In [ ]:
@torch.no_grad()
def encode_pool_np(model, feed, batch=256):
    model.eval(); lk = LOOKUP[feed]; ids = POOL_IDS[feed]; out = []
    for i in range(0, len(ids), batch):
        x = torch.stack([encode_pad(lk[d]) for d in ids[i:i+batch]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out, 0).astype(np.float32)

EMB = {}
for enc_label, model in ENCODERS:
    for feed in FEEDS:
        EMB[(enc_label, feed)] = encode_pool_np(model, feed)
        print(f'  embedded {enc_label} x {feed:<4} -> {EMB[(enc_label, feed)].shape}')

## 10. AUROC (decision 2c) — exhaustive, via streaming histograms

Positives = all pairs with true normLev >= 0.70. Two negative sets:
- **random** (primary, matches the slide): all pairs < 0.70.
- **hard** (stress test): pairs with true normLev in [0.30, 0.70).

Predicted similarity is `1 - ||e_a - e_b|| / 2` from the frozen encoder (the head is discarded, as at
retrieval). All ~55M ordered off-diagonal pairs per feed are streamed in blocks into predicted-sim
histograms (no subsampling, no per-pair storage). AUROC is the Mann-Whitney statistic from the
histograms (0.5 credit for same-bin ties).

In [ ]:
NBINS = 4000; LO, HI = -0.10, 1.10
def _binidx(pred):
    bi = np.floor((pred - LO) / (HI - LO) * NBINS).astype(np.int64)
    return np.clip(bi, 0, NBINS - 1)

def _auroc_from_hist(h_pos, h_neg):
    P, Nn = h_pos.sum(), h_neg.sum()
    if P == 0 or Nn == 0: return float('nan'), int(P), int(Nn)
    cum_below = np.cumsum(h_neg) - h_neg
    auc = float((h_pos * (cum_below + 0.5 * h_neg)).sum() / (P * Nn))
    return auc, int(P), int(Nn)

def auroc_allpairs(feed, block=1024):
    # One pass over the feed's full pairwise grid; accumulate predicted-sim histograms per encoder.
    lk = LOOKUP[feed]; ids = POOL_IDS[feed]; seqs = [lk[d] for d in ids]
    lens = np.array([len(s) for s in seqs]); N = len(ids)
    H = {enc: {'pos': np.zeros(NBINS), 'rand': np.zeros(NBINS), 'hard': np.zeros(NBINS)}
         for enc, _ in ENCODERS}
    E = {enc: EMB[(enc, feed)] for enc, _ in ENCODERS}
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLevenshtein.distance, workers=-1).astype(np.float64)
        denom = np.maximum(lens[r0:r1][:, None], lens[None, :]); denom[denom == 0] = 1
        true = 1.0 - Dm / denom
        rows = np.arange(r1 - r0); true[rows, np.arange(r0, r1)] = 2.0     # sentinel diagonal (excluded)
        offdiag = true <= 1.0000001
        pos  = offdiag & (true >= BAND_HIGH)
        rand = offdiag & (true < BAND_HIGH)
        hard = offdiag & (true >= BAND_MID) & (true < BAND_HIGH)
        for enc, _ in ENCODERS:
            cos = E[enc][r0:r1] @ E[enc].T
            pred = 1.0 - np.sqrt(np.clip(2.0 - 2.0 * cos, 0.0, None)) / 2.0
            bi = _binidx(pred)
            H[enc]['pos']  += np.bincount(bi[pos],  minlength=NBINS)
            H[enc]['rand'] += np.bincount(bi[rand], minlength=NBINS)
            H[enc]['hard'] += np.bincount(bi[hard], minlength=NBINS)
    return H

AUROC_ROWS = []
print('Computing exhaustive AUROC (streaming histograms over the full pairwise grid)...')
for feed in FEEDS:
    H = auroc_allpairs(feed)
    for enc, _ in ENCODERS:
        a_rand, P, Nr = _auroc_from_hist(H[enc]['pos'], H[enc]['rand'])
        a_hard, _, Nh = _auroc_from_hist(H[enc]['pos'], H[enc]['hard'])
        role = 'in-domain' if (enc, feed) in [('AA-enc', 'AA'), ('SS-enc', 'SS')] else 'cross-rep'
        AUROC_ROWS.append(dict(encoder=enc, feed=feed, role=role, n_pos=P,
                               auroc_random=a_rand, n_neg_random=Nr,
                               auroc_hard=a_hard, n_neg_hard=Nh))
        print(f'  {enc} x {feed:<4} | AUROC(random neg)={a_rand:.4f} | AUROC(hard neg [0.30,0.70))={a_hard:.4f} '
              f'| n_pos={P:,}')
auroc_df = pd.DataFrame(AUROC_ROWS)

## 11. Retrieval MAP@10 at bar 0.70 and 0.90 + length baseline + bootstrap CI

Same metric machinery as 24e. Queries = every domain with >=1 true neighbour at the bar (de-hubbed).
Relevance = the exact-Levenshtein neighbour set at the bar. Length-only baseline ranks by length
closeness (the control from 24e).

In [ ]:
N_BOOT = 2000; K_LIST = (10, 50)

def _ap_at_k(order, true_set, k):
    nt = len(true_set)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for i, oi in enumerate(order[:k], start=1):
        if oi in true_set:
            hits += 1; ap += hits / i
    return ap / min(nt, k)

def _per_query(order_fn, T):
    per = {'ap': [], 'hit10': [], 'rec50': [], 'nt': []}
    for qi, arr in T.items():
        ts = set(arr.tolist()); nt = len(ts)
        if nt == 0: continue
        order = order_fn(qi)
        per['nt'].append(nt)
        per['ap'].append(_ap_at_k(order, ts, K_MAP))
        r10 = len(set(order[:10].tolist()) & ts); r50 = len(set(order[:50].tolist()) & ts)
        per['hit10'].append(1.0 if r10 >= 1 else 0.0)
        per['rec50'].append(r50 / nt)
    return {k: np.asarray(v, float) for k, v in per.items()}

def _summarize(per, seed=SEED):
    rng = np.random.default_rng(seed); n = len(per['nt'])
    out = {'n_q': n, 'median_n_true': float(np.median(per['nt'])) if n else np.nan}
    fields = {'MAP@10': 'ap', 'hitrate@10': 'hit10', 'recall@50': 'rec50'}
    bidx = rng.integers(0, n, size=(N_BOOT, n)) if n >= 2 else None
    for name, kk in fields.items():
        arr = per[kk]; out[name] = float(arr.mean()) if n else np.nan
        if bidx is not None:
            b = arr[bidx].mean(axis=1)
            out[name + '_lo'] = float(np.percentile(b, 2.5)); out[name + '_hi'] = float(np.percentile(b, 97.5))
        else:
            out[name + '_lo'] = out[name + '_hi'] = np.nan
    return out

def enc_order_fn(E):
    def f(qi):
        sim = 1.0 - np.linalg.norm(E - E[qi], axis=1) / 2.0
        sim[qi] = -np.inf
        return np.argsort(-sim)
    return f

def len_order_fn(lens):
    def f(qi):
        sim = 1.0 - np.abs(lens - lens[qi]) / np.maximum(lens, lens[qi])
        sim[qi] = -np.inf
        return np.argsort(-sim)
    return f

BAR_KEY = {'0.70': 'T_high', '0.90': 'T_strict'}
results = []
for bar, tkey in BAR_KEY.items():
    for enc_label, _ in ENCODERS:
        for feed in FEEDS:
            T = ORACLE[feed][tkey]
            per = _per_query(enc_order_fn(EMB[(enc_label, feed)]), T)
            s = _summarize(per)
            role = 'in-domain' if (enc_label, feed) in [('AA-enc', 'AA'), ('SS-enc', 'SS')] else 'cross-rep'
            results.append({'bar': bar, 'encoder': enc_label, 'feed': feed, 'role': role, **s})
            print(f"bar {bar} | {enc_label} x {feed:<4} | MAP@10={s['MAP@10']:.3f} "
                  f"[{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | hit@10={s['hitrate@10']:.3f} "
                  f"| med|T|={s['median_n_true']:.0f} (n_q={s['n_q']})")
    for feed in FEEDS:
        lens = ORACLE[feed]['lens'].astype(float)
        per = _per_query(len_order_fn(lens), ORACLE[feed][tkey])
        s = _summarize(per)
        results.append({'bar': bar, 'encoder': 'length-only', 'feed': feed, 'role': 'baseline', **s})
        print(f"bar {bar} | length   x {feed:<4} | MAP@10={s['MAP@10']:.3f} "
              f"[{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | med|T|={s['median_n_true']:.0f}")
res_df = pd.DataFrame(results)

## 12. Summary tables + CSV

In [ ]:
pd.set_option('display.width', 160)
print('=' * 60, '\nAUROC (2c) — exhaustive\n', '=' * 60, sep='')
print(auroc_df.to_string(index=False))
auroc_df.to_csv('colab24f_auroc.csv', index=False)

for bar in ['0.70', '0.90']:
    print('\n' + '=' * 60, f'\nRETRIEVAL @ bar {bar}\n', '=' * 60, sep='')
    sub = res_df[res_df['bar'] == bar][['encoder', 'feed', 'role', 'MAP@10', 'MAP@10_lo',
                                        'MAP@10_hi', 'hitrate@10', 'recall@50', 'median_n_true', 'n_q']]
    print(sub.to_string(index=False))
res_df.to_csv('colab24f_retrieval.csv', index=False)
print('\nSaved colab24f_auroc.csv + colab24f_retrieval.csv')

## 13. Chart 1 — AUROC (random-negative headline; hard-negative annotated). Presentation style.

In [ ]:
import matplotlib.pyplot as plt
COL_AA_ENC = '#1f77b4'; COL_SS_ENC = '#2ca02c'
def _av(enc, col):
    return [float(auroc_df[(auroc_df.encoder == enc) & (auroc_df.feed == f)][col].values[0]) for f in FEEDS]
x = np.arange(len(FEEDS)); w = 0.38
fig, ax = plt.subplots(figsize=(8.8, 5.4))
b1 = ax.bar(x - w/2, _av('AA-enc', 'auroc_random'), w, label='AA-encoder', color=COL_AA_ENC)
b2 = ax.bar(x + w/2, _av('SS-enc', 'auroc_random'), w, label='SS-encoder', color=COL_SS_ENC)
ax.axhline(0.5, ls='--', color='grey', lw=1)
ax.bar_label(b1, fmt='%.3f', padding=2, fontsize=9); ax.bar_label(b2, fmt='%.3f', padding=2, fontsize=9)
# annotate hard-negative AUROC under each bar
for i, f in enumerate(FEEDS):
    ha = float(auroc_df[(auroc_df.encoder == 'AA-enc') & (auroc_df.feed == f)]['auroc_hard'].values[0])
    hs = float(auroc_df[(auroc_df.encoder == 'SS-enc') & (auroc_df.feed == f)]['auroc_hard'].values[0])
    np_ = int(auroc_df[auroc_df.feed == f]['n_pos'].values[0])
    ax.annotate(f'hard-neg\nAA {ha:.3f} / SS {hs:.3f}\nn_pos={np_:,}', (i, 0), (0, -30),
                textcoords='offset points', ha='center', va='top', fontsize=7.5, color='dimgray')
ax.set_ylim(0, 1.08); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel('AUROC (is-high >= 0.70, random negatives)'); ax.set_xlabel('Feed modality')
ax.set_xticks(x); ax.set_xticklabels(FEEDS)
ax.set_title('Exhaustive AUROC — high vs random pair (bars); hard-negative [0.30,0.70) below')
ax.legend(loc='lower left', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab24f_auroc.png', dpi=150, bbox_inches='tight'); plt.show()

## 14. Chart 2 — MAP@10 vs length baseline at bar 0.70 (comparison, matches slide 17)

In [ ]:
def _map_ci(bar, enc):
    d = res_df[(res_df.bar == bar) & (res_df.encoder == enc)]
    p  = np.array([float(d[d.feed == f]['MAP@10'].values[0]) for f in FEEDS])
    lo = np.array([float(d[d.feed == f]['MAP@10_lo'].values[0]) for f in FEEDS])
    hi = np.array([float(d[d.feed == f]['MAP@10_hi'].values[0]) for f in FEEDS])
    return p, lo, hi

def _map_chart(bar, title):
    x = np.arange(len(FEEDS)); w = 0.27
    fig, ax = plt.subplots(figsize=(9.6, 5.5))
    for off, enc, col, lab in [(-w, 'AA-enc', COL_AA_ENC, 'AA-encoder'),
                               (0.0, 'SS-enc', COL_SS_ENC, 'SS-encoder'),
                               (w, 'length-only', '#888888', 'length-only baseline')]:
        p, lo, hi = _map_ci(bar, enc)
        ax.bar(x + off, p, w, label=lab, color=col)
        ax.errorbar(x + off, p, yerr=np.vstack([p - lo, hi - p]), fmt='none', ecolor='black', capsize=3, lw=1)
    ax.set_ylim(0, 1.05); ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_ylabel(f'MAP@{K_MAP} (vs exact-Levenshtein neighbour set)'); ax.set_xlabel('Feed modality')
    ax.set_xticks(x); ax.set_xticklabels(FEEDS)
    ax.set_title(title)
    for i, f in enumerate(FEEDS):
        mt = res_df[(res_df.bar == bar) & (res_df.encoder == 'AA-enc') & (res_df.feed == f)]['median_n_true'].values[0]
        nq = res_df[(res_df.bar == bar) & (res_df.encoder == 'AA-enc') & (res_df.feed == f)]['n_q'].values[0]
        ax.annotate(f'med|T|={mt:.0f}\nn_q={nq:.0f}', (i, 0), (0, -28), textcoords='offset points',
                    ha='center', va='top', fontsize=8, color='dimgray')
    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
    plt.tight_layout(); plt.savefig(f'colab24f_map_bar{bar.replace(".","")}.png', dpi=150, bbox_inches='tight'); plt.show()

_map_chart('0.70', 'Retrieval MAP@10 vs length baseline — bar 0.70 (all-pairs, de-hubbed; cf. slide 17)')

## 15. Chart 3 — MAP@10 at bar 0.90 (well-posed panel, SS/3Di)

In [ ]:
_map_chart('0.90', 'Retrieval MAP@10 vs length baseline — bar 0.90 (well-posed: near-identical only)')

## How to read this notebook

- **§7 all-pairs oracle** replaces 24e's supplied-file pair selection: exact normLev over the entire
  filtered pool, true-neighbour sets at 0.70 and 0.90, query set = every domain with a neighbour.
- **§8 comparison table** is the receipt for *why* 24f differs from 24e: the sample surfaced a tiny,
  hub-biased slice (SS sampled-query med|T|=434 vs representative 22); AA sample == exhaustive.
- **§10 AUROC (2c)** exhaustive, two negative sets (random = slide framing; hard = [0.30,0.70) stress
  test). Uses the encoder L2 similarity; the head is discarded, as at retrieval.
- **§11 retrieval** at bar **0.70** (comparison, de-hubbed) and **0.90** (well-posed). Where an encoder
  MAP@10 bar clears the grey length-only bar with non-overlapping CIs, the embedding contributes
  sequence-order signal beyond length; where they overlap, that feed is length-explained.
- **Consistency check:** AA numbers here must match 24e (AA all-pairs == 24e's AA eval). SS/3Di are the
  de-biased upgrade. Model, pools, training identical to 24e — read differences as the eval change only.
- **No biological claim is made**; relevance is the exact global-Levenshtein neighbour set.